<a href="https://colab.research.google.com/github/Gagangowdatg/BIS_1BM24CS098/blob/main/Particle%20Swarm%20Optimization%20(PSO).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# ==========================================
# STEP 1: DEFINE THE PROBLEM
# ==========================================
NUM_NURSES = 5
NUM_DAYS = 7
NUM_SHIFTS = 3  # 0: Morning, 1: Evening, 2: Night
DIMENSIONS = NUM_NURSES * NUM_DAYS * NUM_SHIFTS  # 105 binary decisions

def fitness_function(binary_position):
    """
    Calculates penalty score for a schedule. Lower score = better schedule.
    """
    # Reshape 1D array into (Nurses, Days, Shifts)
    schedule = binary_position.reshape((NUM_NURSES, NUM_DAYS, NUM_SHIFTS))
    penalty = 0

    # Constraint 1: Every shift on every day needs at least 2 nurses
    nurses_per_shift = np.sum(schedule, axis=0)  # Shape: (7, 3)
    for d in range(NUM_DAYS):
        for s in range(NUM_SHIFTS):
            if nurses_per_shift[d, s] < 2:
                penalty += (2 - nurses_per_shift[d, s]) * 10  # Heavy penalty

    # Constraint 2: A nurse should work at most 1 shift per day
    shifts_per_day = np.sum(schedule, axis=2)  # Shape: (5, 7)
    for n in range(NUM_NURSES):
        for d in range(NUM_DAYS):
            if shifts_per_day[n, d] > 1:
                penalty += (shifts_per_day[n, d] - 1) * 8

    # Constraint 3: A nurse should work at most 5 shifts per week
    weekly_shifts = np.sum(schedule, axis=(1, 2))  # Shape: (5,)
    for n in range(NUM_NURSES):
        if weekly_shifts[n] > 5:
            penalty += (weekly_shifts[n] - 5) * 5

    return penalty

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -10, 10)))

# ==========================================
# STEP 2: INITIALIZE PARAMETERS
# ==========================================
N = 40          # Number of particles (candidate schedules)
w = 0.7         # Inertia weight
c1 = 1.5        # Cognitive coefficient (personal best weight)
c2 = 1.5        # Social coefficient (global best weight)
T = 150         # Maximum iterations

# ==========================================
# STEP 3: INITIALIZE PARTICLES
# ==========================================
np.random.seed(42)
# Continuous velocities and binary positions (0 = Off, 1 = Working)
velocities = np.random.uniform(-1, 1, (N, DIMENSIONS))
positions = np.random.randint(0, 2, (N, DIMENSIONS))

# Track personal bests (pbest) and global best (gbest)
pbest_positions = positions.copy()
pbest_scores = np.array([fitness_function(p) for p in positions])

gbest_index = np.argmin(pbest_scores)
gbest_position = pbest_positions[gbest_index].copy()
gbest_score = pbest_scores[gbest_index]

# ==========================================
# STEPS 4, 5 & 6: EVALUATE, UPDATE & ITERATE
# ==========================================
for iteration in range(T):
    for i in range(N):
        # Step 4: Evaluate Fitness
        current_score = fitness_function(positions[i])

        # Update Personal Best
        if current_score < pbest_scores[i]:
            pbest_scores[i] = current_score
            pbest_positions[i] = positions[i].copy()

            # Update Global Best
            if current_score < gbest_score:
                gbest_score = current_score
                gbest_position = positions[i].copy()

        # Step 5: Update Velocities and Positions
        r1 = np.random.rand(DIMENSIONS)
        r2 = np.random.rand(DIMENSIONS)

        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest_positions[i] - positions[i])
            + c2 * r2 * (gbest_position - positions[i])
        )

        # Convert continuous velocity into binary schedule (0 or 1) via Sigmoid
        probabilities = sigmoid(velocities[i])
        positions[i] = (np.random.rand(DIMENSIONS) < probabilities).astype(int)

    # Stop early if a zero-penalty (perfect) schedule is found
    if gbest_score == 0:
        print(f"Optimal schedule found at iteration {iteration + 1}!")
        break

# ==========================================
# STEP 7: OUTPUT THE BEST SOLUTION
# ==========================================
print(f"\nFinal Best Penalty Score: {gbest_score}")
best_schedule = gbest_position.reshape((NUM_NURSES, NUM_DAYS, NUM_SHIFTS))

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
shifts = ["Morning", "Evening", "Night"]

print("\n--- Optimized Weekly Staff Schedule ---")
for d_idx, day in enumerate(days):
    print(f"\n{day}:")
    for s_idx, shift in enumerate(shifts):
        assigned_nurses = [
            f"Nurse {n+1}" for n in range(NUM_NURSES) if best_schedule[n, d_idx, s_idx] == 1
        ]
        print(f"  {shift:<8}: {', '.join(assigned_nurses) if assigned_nurses else 'None'}")


Final Best Penalty Score: 182

--- Optimized Weekly Staff Schedule ---

Mon:
  Morning : Nurse 3
  Evening : Nurse 2
  Night   : Nurse 1, Nurse 4

Tue:
  Morning : Nurse 1, Nurse 4
  Evening : None
  Night   : Nurse 5

Wed:
  Morning : Nurse 5
  Evening : Nurse 1, Nurse 2
  Night   : Nurse 3

Thu:
  Morning : Nurse 2, Nurse 5
  Evening : Nurse 4, Nurse 5
  Night   : Nurse 1, Nurse 2, Nurse 3

Fri:
  Morning : Nurse 3
  Evening : Nurse 1, Nurse 4
  Night   : Nurse 1, Nurse 2

Sat:
  Morning : Nurse 2, Nurse 3
  Evening : Nurse 3, Nurse 4
  Night   : Nurse 1, Nurse 5

Sun:
  Morning : Nurse 5
  Evening : Nurse 2
  Night   : None
